# Raw cavity waveforms of the DD captures

One dataframe per waveform id, one row per capture, cycle and cavity: the whole waveform as an array, with its
mean, standard deviation and peak to peak (`max_min`). 43 DD captures, 81 cavities, about 35,400 rows per
waveform id.

The parquets are written on DALIA by `ess_dd_waveforms.py` (`sbatch ess_dd_waveforms.sh`, 4 min) to
`/lustre/work/pdl18409/udl259b52/ess_dd/out_waveforms/`. Empty records are left out. Record lengths differ by
section (5,032 to 35,221 samples), so `waveform` is an array of varying length.

| Waveform id | Dataframe | Signal |
|---|---|---|
| `Dwn0-Cmp0` | `field_amp` | cavity field, amplitude |
| `Dwn0-Cmp1` | `field_phase` | cavity field, phase (degrees) |
| `Dwn5-Cmp0` | `fwd_amp` | forward power, amplitude |
| `Dwn5-Cmp1` | `fwd_phase` | forward power, phase (degrees) |
| `Dwn6-Cmp0` | `refl_amp` | reflected power, amplitude |
| `Dwn6-Cmp1` | `refl_phase` | reflected power, phase (degrees) |


In [2]:
from pathlib import Path

import pandas as pd

# Kernels see the DALIA root under /workspace.
OUT = Path("/workspace/lustre/work/pdl18409/udl259b52/ess_dd/out_waveforms")

dfs = {p.stem: pd.read_parquet(p) for p in sorted(OUT.glob("*.parquet"))}
field_amp, field_phase = dfs["field_amp"], dfs["field_phase"]
fwd_amp, fwd_phase = dfs["fwd_amp"], dfs["fwd_phase"]
refl_amp, refl_phase = dfs["refl_amp"], dfs["refl_phase"]
pd.DataFrame({name: {"waveform_id": df.waveform_id.iat[0], "rows": len(df), "cavities": df.cavity.nunique(),
                     "captures": df.event_id.nunique(), "min samples": df.waveform.str.len().min(),
                     "max samples": df.waveform.str.len().max()} for name, df in dfs.items()}).T

,waveform_id,rows,cavities,captures,min samples,max samples
field_amp,Dwn0-Cmp0,35414,81,43,5032,35221
field_phase,Dwn0-Cmp1,35413,81,43,5032,35221
fwd_amp,Dwn5-Cmp0,35419,81,43,5032,35221
fwd_phase,Dwn5-Cmp1,35419,81,43,5032,35221
refl_amp,Dwn6-Cmp0,35419,81,43,5032,35221
refl_phase,Dwn6-Cmp1,35419,81,43,5032,35221


In [1]:
field_amp.head(30)

NameError: name 'field_amp' is not defined

## Mean, std and max - min per section

Median over the rows of each section. The section is the start of the cavity name.

In [3]:
pd.concat({name: df.groupby(df.cavity.str.split("-").str[0])[["mean", "std", "max_min"]].median()
           for name, df in dfs.items()}).round(3)

mean      std   max_min
            cavity                            
field_amp   HBL      10.102    8.796    19.373
            MBL       7.147    6.091    13.629
            Spk       4.664    4.341     9.284
field_phase HBL     -12.479   14.755   339.533
            MBL     -18.942   17.629   335.476
            Spk      23.909   79.491   359.372
fwd_amp     HBL     162.458  212.425   889.471
            MBL      85.770  120.688   493.317
            Spk      53.286   58.614   240.236
fwd_phase   HBL     -12.228   90.781   357.385
            MBL      18.148   84.075   355.595
            Spk      11.097   88.824   357.798
refl_amp    HBL     154.114  164.202  1241.273
            MBL      81.411   83.853   508.860
            Spk      51.076   53.794   360.878
refl_phase  HBL      34.704   49.917   359.483
            MBL       7.523   48.146   359.574
            Spk      20.297   86.599   359.840